# UAV Deforestation Detection — Reinforcement Learning

**Problem:** Forests face deforestation from human activity — trees being cut, land clearing, environmental degradation. Traditional monitoring cannot cover large areas continuously.

**Solution:** Train an AI drone (UAV) to patrol a 15x15 forest grid, detect deforestation, and deploy retardant spray to stop it spreading.

**Each step (1 of 300):**
1. Drone sees 5x5 area (partial observation)
2. Picks 1 of 8 actions: Move, Scan, Deploy Retardant, Call Rangers, Return to Base
3. Forest evolves: deforestation spreads, new threats appear
4. Rewards: +50 retardant, +60 rangers, -0.5 per idle step
5. **WIN** if health >= 70% at step 300

**4 Algorithms:** DQN, REINFORCE, PPO, A2C

## 1. Setup

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

from environment.forest_env import (
    DeforestationEnv, HEALTHY, AT_RISK, DEFORESTING, DESTROYED,
    GRID_SIZE, MAX_FUEL, MAX_STEPS
)

print(f'Grid: {GRID_SIZE}x{GRID_SIZE} | Steps: {MAX_STEPS} | Fuel: {MAX_FUEL}')
env = DeforestationEnv()
print(f'Observation: {env.observation_space.shape} | Actions: {env.action_space.n}')
env.close()

## 2. Environment

**15x15 grid** with 4 tile states:
- **Healthy** (green) — forest intact
- **At-Risk** (yellow) — early warning
- **Deforesting** (orange) — actively being destroyed, spreads to neighbors
- **Destroyed** (brown) — gone, stumps remain

**Dynamics:** Wind changes every 8 steps (spread faster downwind), rain slows spread (35% chance every 10 steps), new threats every 50 steps, retardant protects for 15 steps.

In [ ]:
def plot_grid(grid, drone_pos, title='Forest Grid'):
    colors = {0: [0.18,0.54,0.18], 1: [0.78,0.72,0.12], 2: [0.86,0.39,0.08], 3: [0.35,0.18,0.06]}
    img = np.zeros((GRID_SIZE, GRID_SIZE, 3))
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            img[r,c] = colors[grid[r,c]]
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    ax.imshow(img, interpolation='nearest')
    ax.plot(drone_pos[1], drone_pos[0], 'co', markersize=15, label='Drone')
    ax.set_title(title, fontsize=14); ax.legend(); ax.set_xticks([]); ax.set_yticks([])
    plt.show()

env = DeforestationEnv()
obs, info = env.reset(seed=42)
plot_grid(info['grid'], info['drone_pos'], 'Initial Forest State')
env.close()

## 3. Reward Structure

| Event | Reward | Purpose |
|-------|--------|---------|
| Each step | -0.5 | Efficiency pressure |
| Retardant on Deforesting | **+50** | Active intervention |
| Retardant on At-Risk | +30 | Prevention |
| Call Rangers | **+60** | Threat interception |
| Scan finds Deforesting | +15 | Detection |
| Win (health >= 70%) | +100 | Mission success |
| Lose (health < 30%) | -100 | Mission failure |

## 4. The Four RL Algorithms

### DQN (Value-Based)
Learns Q(s,a) = expected future reward. **Replay buffer** (100k memories) stores past experiences. **Target network** updated every 500 steps for stability. Epsilon-greedy exploration.

### REINFORCE (Policy Gradient)
Directly learns action probabilities. Waits until episode END to update (Monte Carlo). No memory, no critic. Simplest but highest variance.

### PPO (Proximal Policy Optimization)
Like REINFORCE but **clips policy changes to max 20%** per update. Stable but can get trapped in local optima.

### A2C (Actor-Critic)
Two networks: **Actor** picks actions, **Critic** evaluates states. Updates every **32 steps** (not waiting for episode end). Fastest feedback.

## 5. Load & Evaluate Models

In [ ]:
from stable_baselines3 import DQN, PPO, A2C
from agents.reinforce_agent import ReinforceAgent

ACTION_NAMES = ['Up','Down','Left','Right','Scan','Retardant','Rangers','Base']

def evaluate_model(model, label, is_rf=False, n=20):
    env = DeforestationEnv()
    wins, healths, rews, ac = 0, [], [], {a:0 for a in range(8)}
    for ep in range(n):
        obs, _ = env.reset(seed=ep+500)
        tr = 0
        while True:
            a, _ = model.predict(obs, deterministic=True)
            if not is_rf: a = int(a)
            ac[a] += 1
            obs, r, t, trunc, info = env.step(a)
            tr += r
            if t or trunc:
                healths.append(info['forest_health'])
                rews.append(tr)
                if info['forest_health'] >= 0.7: wins += 1
                break
    env.close()
    total = sum(ac.values())
    print(f'{label}: {wins}/{n} wins ({wins/n*100:.0f}%) | Health: {np.mean(healths):.1%} | Reward: {np.mean(rews):.0f}')
    for a in range(8):
        print(f'  {ACTION_NAMES[a]:12s} {ac[a]/total*100:5.1f}%')
    return {'win_rate': wins/n, 'mean_health': float(np.mean(healths)),
            'mean_reward': float(np.mean(rews)),
            'action_dist': {ACTION_NAMES[a]: ac[a]/total for a in range(8)}}

In [ ]:
results = {}

# A2C
p = os.path.join(PROJECT_ROOT, 'models/best_model')
if os.path.exists(p+'.zip'):
    print('Loading A2C...'); results['A2C'] = evaluate_model(A2C.load(p), 'A2C')
print()

# DQN
p = os.path.join(PROJECT_ROOT, 'models/retrained/dqn_retrained')
if os.path.exists(p+'.zip'):
    print('Loading DQN...'); results['DQN'] = evaluate_model(DQN.load(p), 'DQN')
print()

# PPO
p = os.path.join(PROJECT_ROOT, 'models/retrained/ppo_retrained')
if os.path.exists(p+'.zip'):
    print('Loading PPO...'); results['PPO'] = evaluate_model(PPO.load(p), 'PPO')
print()

# REINFORCE
rf_path = os.path.join(PROJECT_ROOT, 'models/retrained/reinforce_retrained.pt')
env = DeforestationEnv()
rf = ReinforceAgent(env.observation_space.shape[0], env.action_space.n)
env.close()
if os.path.exists(rf_path):
    rf.load(rf_path); print('Loading REINFORCE...')
    results['REINFORCE'] = evaluate_model(rf, 'REINFORCE', is_rf=True)

## 6. Comparison Plots

In [ ]:
algos = [a for a in ['A2C','DQN','PPO','REINFORCE'] if a in results]
clrs = {'A2C':'#3fb950','DQN':'#58a6ff','PPO':'#f0883e','REINFORCE':'#f85149'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Algorithm Comparison', fontsize=14, fontweight='bold')

# Win Rate
ax = axes[0][0]
wr = [results[a]['win_rate']*100 for a in algos]
bars = ax.bar(algos, wr, color=[clrs[a] for a in algos])
ax.set_ylabel('Win Rate (%)'); ax.set_title('Win Rate'); ax.set_ylim(0,115)
for b,v in zip(bars,wr): ax.text(b.get_x()+b.get_width()/2, b.get_height()+2, f'{v:.0f}%', ha='center', fontweight='bold')

# Health
ax = axes[0][1]
mh = [results[a]['mean_health']*100 for a in algos]
bars = ax.bar(algos, mh, color=[clrs[a] for a in algos])
ax.set_ylabel('Health (%)'); ax.set_title('Mean Forest Health')
ax.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='Win=70%'); ax.legend(); ax.set_ylim(0,100)
for b,v in zip(bars,mh): ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{v:.1f}%', ha='center', fontsize=9)

# Reward
ax = axes[1][0]
mr = [results[a]['mean_reward'] for a in algos]
bars = ax.bar(algos, mr, color=[clrs[a] for a in algos])
ax.set_ylabel('Reward'); ax.set_title('Mean Reward')
ax.axhline(y=0, color='grey', linestyle='--', alpha=0.3)

# Action Distribution
ax = axes[1][1]
x = np.arange(len(ACTION_NAMES)); w = 0.2
for i,algo in enumerate(algos):
    dist = [results[algo]['action_dist'].get(a,0)*100 for a in ACTION_NAMES]
    ax.bar(x+i*w, dist, w, label=algo, color=clrs[algo], alpha=0.8)
ax.set_ylabel('Usage (%)'); ax.set_title('Action Distribution')
ax.set_xticks(x+w*1.5); ax.set_xticklabels(ACTION_NAMES, rotation=45, ha='right'); ax.legend()

plt.tight_layout()
plt.savefig('analysis/algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Best Agent in Action

In [ ]:
if 'A2C' in results:
    env = DeforestationEnv()
    obs, info = env.reset(seed=42)
    model = A2C.load(os.path.join(PROJECT_ROOT, 'models/best_model'))
    snaps, total_r = [], 0
    for step in range(MAX_STEPS):
        a, _ = model.predict(obs, deterministic=True)
        obs, r, t, tr, info = env.step(int(a))
        total_r += r
        if step in [0,50,100,150,200,250,299] or t or tr:
            snaps.append((step, info.copy(), total_r))
        if t or tr: break
    env.close()
    cm = {0:[0.18,0.54,0.18],1:[0.78,0.72,0.12],2:[0.86,0.39,0.08],3:[0.35,0.18,0.06]}
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for idx,(step,info,rew) in enumerate(snaps[:8]):
        ax = axes[idx//4][idx%4]
        img = np.zeros((GRID_SIZE,GRID_SIZE,3))
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE): img[r,c] = cm[info['grid'][r,c]]
        ax.imshow(img); ax.plot(info['drone_pos'][1], info['drone_pos'][0], 'co', ms=10)
        ax.set_title(f'Step {step} | H={info["forest_health"]:.0%}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    for idx in range(len(snaps),8): axes[idx//4][idx%4].axis('off')
    fig.suptitle('A2C Episode Progression', fontweight='bold')
    plt.tight_layout(); plt.show()

## 8. Analysis

### A2C Won (100%)
Critic provides feedback every 32 steps = ~9 updates per episode. Actor quickly learned: move toward threats, deploy retardant, move to next.

### DQN Strong (80%)
Replay buffer stores 100k experiences. Rare +50 retardant events get replayed hundreds of times.

### PPO Struggles (10%)
Clipped objective limits changes to 20%. Got trapped doing 76% retardant. Could not shift back to exploration. Classic local optimum.

### REINFORCE Fails (5%)
Waits 300 steps to update. Cannot tell which of 300 actions mattered. Monte Carlo return = one noisy number for entire episode.

## 9. Conclusion

| Algorithm | Win Rate | Key Insight |
|-----------|----------|-------------|
| **A2C** | **100%** | Fast critic feedback enables reactive strategy |
| **DQN** | **80%** | Replay buffer learns from rare events |
| **PPO** | **10%** | Clipping traps in local optimum |
| **REINFORCE** | **5%** | 300-step credit assignment impossible |

**Finding:** Algorithms with fast feedback (A2C) or memory (DQN) outperform conservative (PPO) or delayed (REINFORCE) methods in complex environments.